# 01 · Census Income (contest exercise)

**Style C: contest simulation.** Structure copied from the IOAI 2025 notebooks (T25-ANTIQUE,
T25-G-SPEECH): statement, seed cell, data loading, a weak baseline, local validation, a submission
cell and an official grader. **Traces to** `notes/ioai-task-patterns.md` P1, P2, P3.

## 1. Problem Description

A regional statistics office wants to flag households likely to earn more than $50K a year for a
targeted survey. Surveyors can only visit a limited number of homes, and both kinds of error are costly, so the
office cares equally about finding high earners and about correctly clearing low earners.

## 2. Dataset

The data is the 1994 US census extract (OpenML `adult`, version 2), loaded and split by the cell below.

| Split | Rows | Labels? | Used for |
|---|---|---|---|
| `train` | 30,000 | ✅ `target` column | training and your own validation |
| `test_a` | 9,000 | hidden | Leaderboard A (public, check as often as you like) |
| `test_b` | 9,842 | hidden | Leaderboard B (**final score**) |

14 features: 6 numeric, 8 categorical (`category` dtype). `workclass`, `occupation` and `native-country`
have missing values.

## 3. Task

Predict `target` (1 = income > 50K) for every row of `test_a` and `test_b`.

## 4. Submission

Produce `submission.zip` containing:
- `submissionA.csv`: predictions for `test_a`, **one 0/1 per line, no header**, in the original row order.
- `submissionB.csv`: the same for `test_b`.

A file with a header, the wrong length or values outside {0, 1} scores 0.

## 5. Scoring

**Macro F1** (the mean of the F1 for class 0 and the F1 for class 1).

- Baseline score (Leaderboard B): **0.69** (logistic regression on the numeric columns, below)
- Reference score (Leaderboard B): **0.81**, which is your target

## 6. Requirements

- The whole notebook must run top to bottom in **≤ 5 minutes on CPU** (e.g. free Colab), training included.
- Libraries: numpy, pandas, scikit-learn, xgboost, lightgbm. No external data, no pretrained models.
- Never read the hidden labels. The grader cell at the bottom is the organisers' server.

In [ ]:
import time, os, zipfile, random
T0 = time.time()
TIME_LIMIT_MIN = 5

import numpy as np
import pandas as pd

seed = 42
random.seed(seed)
np.random.seed(seed)

### Data loading (DO NOT CHANGE)

In [ ]:
from sklearn.datasets import fetch_openml

def load_census():
    df = fetch_openml("adult", version=2, as_frame=True).frame.copy()
    df["target"] = (df.pop("class") == ">50K").astype(int)
    idx = np.random.default_rng(2025).permutation(len(df))
    return df, idx

_df, _idx = load_census()
train = _df.iloc[_idx[:30000]].reset_index(drop=True)
test_a = _df.iloc[_idx[30000:39000]].drop(columns="target").reset_index(drop=True)
test_b = _df.iloc[_idx[39000:]].drop(columns="target").reset_index(drop=True)
del _df, _idx          # the organisers keep the answers
print(train.shape, test_a.shape, test_b.shape)
train.head()

### Baseline (replace this)

Logistic regression on the six numeric columns, with a 0.5 threshold.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

NUM = ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week"]

def fit_predict(train_df, test_dfs):
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    model.fit(train_df[NUM], train_df["target"])
    return [(model.predict_proba(t[NUM])[:, 1] >= 0.5).astype(int) for t in test_dfs]

### Local validation

Hold out part of `train` and score it with the official metric before you spend a submission.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

tr_part, va_part = train_test_split(train, test_size=0.2, stratify=train["target"], random_state=0)
(va_pred,) = fit_predict(tr_part, [va_part.drop(columns="target")])
print(f"local macro F1: {f1_score(va_part['target'], va_pred, average='macro'):.4f}")

### Submission

In [ ]:
pred_a, pred_b = fit_predict(train, [test_a, test_b])

def write_labels(pred, path):
    pd.Series(np.asarray(pred).astype(int)).to_csv(path, index=False, header=False)

write_labels(pred_a, "submissionA.csv")
write_labels(pred_b, "submissionB.csv")
with zipfile.ZipFile("submission.zip", "w") as z:
    z.write("submissionA.csv"); z.write("submissionB.csv")

# format check (the platform would reject a malformed zip)
with zipfile.ZipFile("submission.zip") as z:
    for name, n in [("submissionA.csv", len(test_a)), ("submissionB.csv", len(test_b))]:
        lines = z.read(name).decode().splitlines()
        assert len(lines) == n and set(lines) <= {"0", "1"}, f"{name} malformed"
print("submission.zip written and format-checked")

### Official grader (run it, don't read it)

Prints Leaderboard A. Set `SHOW_FINAL = True` only once, when you have finished.

In [ ]:
SHOW_FINAL = False or os.environ.get("IOAI_SHOW_FINAL") == "1"

def _grade(zip_path="submission.zip"):
    from sklearn.metrics import f1_score as _f1
    df, idx = load_census()
    truth = {"submissionA.csv": df["target"].to_numpy()[idx[30000:39000]],
             "submissionB.csv": df["target"].to_numpy()[idx[39000:]]}
    out = {}
    with zipfile.ZipFile(zip_path) as z:
        for name, y in truth.items():
            try:
                p = np.array([int(v) for v in z.read(name).decode().splitlines()])
                out[name] = _f1(y, p, average="macro") if len(p) == len(y) and set(p) <= {0, 1} else 0.0
            except Exception:
                out[name] = 0.0
    return out["submissionA.csv"], out["submissionB.csv"]

score_a, score_b = _grade()
print(f"Leaderboard A (public): {score_a:.4f}")
print(f"Leaderboard B (final):  {score_b:.4f}" if SHOW_FINAL else "Leaderboard B: hidden until SHOW_FINAL = True")

In [ ]:
elapsed = time.time() - T0
print(f"total runtime {elapsed / 60:.1f} min (limit {TIME_LIMIT_MIN} min)")
assert elapsed < TIME_LIMIT_MIN * 60, "over the time limit: on the platform this run would score 0"